# 🚀 Customer Churn Prediction: Training & Registration (Colab Edition)
This notebook automates the "Excellent" rubric training pipeline for our Capstone project.

### Features:
1. **Kaggle Integration**: Auto-downloads the real 440k dataset.
2. **Hyperparameter Tuning**: RandomizedSearchCV for best model performance.
3. **Rich Visual Reporting**: Confusion Matrix, ROC-AUC, for Model Performance.
4. **Responsible AI**: Logs SHAP plots, Fairness Bar Charts, and Bias metrics to MLflow.
5. **DagsHub Registry**: Registers the model to our cloud dashboard.

In [ ]:
# 1. Install Dependencies
!pip install mlflow dagshub shap deepchecks pandas scikit-learn matplotlib opendatasets seaborn

In [ ]:
# 2. DagsHub & MLflow Configuration
import dagshub
import mlflow
import os

REPO_OWNER = "nhannhb92"
REPO_NAME = "msa24-ddm501-group6-final-project"

dagshub.init(repo_owner=REPO_OWNER, repo_name=REPO_NAME, mlflow=True)
mlflow.set_tracking_uri(f"https://dagshub.com/{REPO_OWNER}/{REPO_NAME}.mlflow")
print("✅ DagsHub Connection Active")

In [ ]:
# 3. Download Real Dataset from Kaggle
import opendatasets as od
import json
import os

# Set Kaggle credentials for automated download
# You can also upload your .secret file to Colab to automate this!
KAGGLE_USER = "bchnhnnguynhunh"
KAGGLE_KEY = "KGAT_5bfb1c6932ba6cb5ec8a3d1c2b84d801"

with open("kaggle.json", "w") as f:
    json.dump({"username": KAGGLE_USER, "key": KAGGLE_KEY}, f)

dataset_url = "https://www.kaggle.com/datasets/muhammadshahidazeem/customer-churn-dataset"
od.download(dataset_url)

# Move files to expected path
!mkdir -p data
!mv customer-churn-dataset/*.csv data/
print("✅ Dataset ready in data/ folder")

In [ ]:
# 4. Run Training Pipeline (Reuse scripts/train_model.py logic)
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc, f1_score
from mlflow.models import infer_signature
import matplotlib.pyplot as plt
import seaborn as sns
import shap

# --- CONFIG ---
DATA_PATH = "data/customer_churn_dataset-training-master.csv"
MODEL_NAME = "CustomerChurnModel"

def train():
    with mlflow.start_run(run_name="Colab_Hyperparameter_Tuning"):
        # 1. Load
        print("Reading data...")
        df = pd.read_csv(DATA_PATH).dropna(subset=['Churn'])
        if 'CustomerID' in df.columns: df = df.drop('CustomerID', axis=1)
        df.columns = [col.lower().replace(' ', '_') for col in df.columns]
        
        X = df.drop('churn', axis=1)
        y = df['churn']
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
        
        # 2. Pipeline
        print("Setting up pipeline...")
        numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
        categorical_features = X.select_dtypes(include=['object']).columns.tolist()
        
        preprocessor = ColumnTransformer(transformers=[
            ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
            ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
        ])
        
        pipeline = Pipeline([('preprocessor', preprocessor), ('classifier', RandomForestClassifier(random_state=42))])
        
        # 3. Tuning
        print("Starting Hyperparameter Tuning...")
        param_dist = {
            'classifier__n_estimators': [100, 300],
            'classifier__max_depth': [10, 20, None],
            'classifier__min_samples_split': [2, 10]
        }
        
        search = RandomizedSearchCV(pipeline, param_dist, n_iter=5, cv=3, scoring='f1', verbose=1)
        search.fit(X_train, y_train)
        best_model = search.best_estimator_
        
        # 4. Metrics & Visualization
        print("Generating Performance Reports...")
        y_pred = best_model.predict(X_test)
        y_probs = best_model.predict_proba(X_test)[:, 1]
        mlflow.log_metrics({"accuracy": (y_pred == y_test).mean(), "f1_score": f1_score(y_test, y_pred)})
        
        # Visual 1: Confusion Matrix
        cm = confusion_matrix(y_test, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm)
        disp.plot(cmap='Blues')
        plt.title("Confusion Matrix")
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        
        # Visual 2: ROC Curve
        fpr, tpr, _ = roc_curve(y_test, y_probs)
        roc_auc = auc(fpr, tpr)
        plt.figure()
        plt.plot(fpr, tpr, label=f'ROC curve (area = {roc_auc:.2f})')
        plt.plot([0, 1], 'k--')
        plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC Curve')
        plt.legend(loc="lower right")
        plt.savefig("roc_curve.png")
        mlflow.log_artifact("roc_curve.png")
        
        # --- RESPONSIBLE AI ---
        print("Running Responsible AI reports...")
        test_copy = X_test.copy()
        test_copy['error'] = np.abs(y_test - y_probs)
        gender_stats = test_copy.groupby(X_test['gender'])['error'].mean()
        gap = np.abs(gender_stats['Male'] - gender_stats['Female'])
        mlflow.log_metric("bias_gap_gender", gap)
        
        # Visual 3: Fairness Chart
        plt.figure()
        sns.barplot(x=gender_stats.index, y=gender_stats.values)
        plt.title("Fairness Analysis: Error by Gender"); plt.ylabel("MAE")
        plt.savefig("fairness_analysis.png")
        mlflow.log_artifact("fairness_analysis.png")
        
        # SHAP Explainer
        explainer_data = X_test.sample(100)
        transformed = best_model.named_steps['preprocessor'].transform(explainer_data)
        explainer = shap.TreeExplainer(best_model.named_steps['classifier'])
        shap_values = explainer.shap_values(transformed)
        shap.summary_plot(shap_values[1], transformed, show=False)
        plt.savefig("shap_summary.png")
        mlflow.log_artifact("shap_summary.png")
        
        # 5. Register
        signature = infer_signature(X_train, best_model.predict(X_train))
        mlflow.sklearn.log_model(best_model, "model", registered_model_name=MODEL_NAME, signature=signature)
        print("🎉 Model trained and registered to DagsHub!")

train()